In [1]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

In [2]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [116]:
class Model:
  def __init__(self, model_name: str = "gpt2"):
    self.model = GPT2LMHeadModel.from_pretrained(model_name)
    self.model.eval()
    self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    self.tokenizer.pad_token = self.tokenizer.eos_token
    self.vocab_size = self.tokenizer.vocab_size
    self.generator = np.random.default_rng(seed=42)

  def greedy_sampling(self, logits: torch.Tensor) -> int:
    val, ind = torch.max(logits, dim=0)
    return ind.item()

  def random_sampling(self, logits: torch.Tensor) -> int:
    probs = torch.softmax(logits, dim=-1).numpy()
    token = self.generator.choice(np.arange(self.vocab_size), p=probs)
    return int(token)

  def _beam_search_generate(
      self,
      prompt: str,
      max_length: int,
      num_beams: int
    ) -> str:
    input_ids = self.tokenizer(prompt, return_tensors="pt")["input_ids"]

    # beam, score, done_flag
    beams = [(input_ids.squeeze(0), 0.0, False)]

    for _ in range(max_length):
      candidates = []
      for beam, score, done in beams:
        if done:
          continue

        with torch.no_grad():
          outputs = self.model(beam.unsqueeze(0))
          next_token_logits = outputs.logits[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)
        candidate_scores = score + torch.log(next_token_probs)

        top_scores, top_ids = torch.topk(candidate_scores, num_beams, sorted=True)

        for i in range(num_beams):
          new_beam = torch.cat([beam, top_ids[i].unsqueeze(0)])
          new_score = top_scores[i]
          new_done = (top_ids[i] == self.tokenizer.eos_token_id)
          candidates.append((new_beam, new_score, new_done))

      sorted_candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
      beams = sorted_candidates[:num_beams]

      if all([beam[2] for beam in beams]):
        break

    best_beam, best_score, best_done = beams[0]
    return self.tokenizer.decode(best_beam, skip_special_tokens=True)

  def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    return logits / temperature

  def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
    if top_p == 0.0 or top_p >= 1.0:
      return logits

    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    cdf = torch.cumsum(sorted_probs, dim=-1)

    mask = cdf <= top_p
    logits_mask = torch.scatter(mask, dim=-1, index=sorted_ids, src=~mask)

    return torch.masked_fill(logits, mask=logits_mask, value=float("-inf"))

  def _apply_top_k(self, logits: torch.Tensor, top_k: float = 0.0) -> torch.Tensor:
    if top_k == 0.0:
      return logits

    top_logits, top_ids = torch.topk(logits, top_k, dim=-1)

    mask = torch.ones_like(logits, dtype=torch.bool)
    mask[top_ids] = False

    return torch.masked_fill(logits, mask=mask, value=float("-inf"))

  def generate(
  self,
  prompt: str,
  max_length: int = 50,
  strategy: str = "greedy",
  temperature: float = 1.0,
  top_k: int = 0,
  top_p: float = 1.0,
  num_beams: int = 3
  ) -> str:
    if strategy == "beam_search":
      return self._beam_search_generate(prompt, max_length, num_beams)

    input_ids = self.tokenizer(prompt, return_tensors="pt")["input_ids"]

    for _ in range(max_length):
      with torch.no_grad():
        outputs = self.model(input_ids)
        next_token_logits = outputs.logits[0, -1, :]

        if strategy == "greedy" or (strategy == "random" and temperature == 0.0):
          next_token = self.greedy_sampling(next_token_logits)
        else:
          next_token_logits = self.apply_temperature(next_token_logits, temperature)

          next_token_logits = self._apply_top_k(next_token_logits, top_k)
          next_token_logits = self._apply_top_p(next_token_logits, top_p)

          next_token = self.random_sampling(next_token_logits)
      input_ids = torch.cat([input_ids, torch.IntTensor([[next_token]])], dim=-1)

      if next_token == self.tokenizer.eos_token_id:
        break
    return self.tokenizer.decode(input_ids[0], skip_special_token=True)

In [117]:
model = Model()

In [118]:
# Продемонстрируйте результат работы `generate` при различных параметрах

def test_generation():
  methods = ["greedy", "random", "beam_search"]
  prompt = "bro what the hell"
  max_length = 50
  temperature = 0.8
  top_k = 10
  top_p = 0.99
  num_beams = 3
  print(prompt)
  for strategy in methods:
    print(f"---{strategy}---")
    answer = model.generate(
        prompt,
        max_length,
        strategy,
        temperature,
        top_k,
        top_p,
        num_beams
    )
    print(f"Answer: {answer}")

In [119]:
test_generation()

bro what the hell
---greedy---
Answer: bro what the hell is going on here?

I'm not sure what to say. I'm not sure what to say.

I'm not sure what to say.

I'm not sure what to say.

I'm not sure what
---random---
Answer: bro what the hell are you talking about. If you're a guy who likes to talk about the things that happen in life, then you've got to be a bit more humble. I've been a little less humble than some of the guys out there, but I
---beam_search---
Answer: bro what the hell is going on here?

I'm not going to lie, I'm not going to lie. I'm not going to lie. I'm not going to lie. I'm not going to lie. I'm not going to lie. I
